# BRCA1 DeltaEmbSAE Interpretability

This notebook inspects the BRCA1 DeltaEmbSAE layer with the best current-run ClinVar AUC. The main view is mutation position on the x axis and sparse-autoencoder feature activation on the y axis, so you can ask which variants and regions activate each learned feature.

The notebook uses the current sequence-deduplicated run label by default and scans benchmark result CSVs directly, so it still works if the global metrics table has not been recollected yet.

In [ ]:
from pathlib import Path
import json
import os
import pickle
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break
sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".matplotlib_cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, spearmanr
from IPython.display import display

RUN_LABEL = "DeltaEmbSAE_max_pool_batchtopk_k64_nf12800_seed42_seqdedup_v2"
DATASET = "MV_BRCA1_Findlay_2018"
OUTPUT_ROOT = REPO_ROOT / "data" / "clinprotgym_esmc_sae"
DATASET_ROOT = OUTPUT_ROOT / "datasets" / DATASET
TABLE_DIR = DATASET_ROOT / "tables"
JOB_DIR = DATASET_ROOT / "jobs"
SEQUENCE_DIR = DATASET_ROOT / "sequence_data"
FIGURE_DIR = DATASET_ROOT / "figures" / "brca1_sae_interpretability"
TABLE_OUT_DIR = TABLE_DIR / "sae_interpretability"
for directory in [FIGURE_DIR, TABLE_OUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Keep these small at first; raise them once you know the notebook is responsive.
TOP_FEATURES_TO_SUMMARIZE = 150
TOP_FEATURES_TO_PLOT = 12
TOP_VARIANTS_PER_FEATURE = 20
SELECTED_FEATURE = None  # Set to an integer feature index after seeing feature_summary.
FEATURE_SELECTION_METRIC = "coverage_score"  # coverage_score, auc_discrimination, spearman_abs, max_activation
CLINICAL_LABEL_MODE = "pipeline"  # pipeline, hgvs, or custom_csv
CUSTOM_CLINICAL_LABELS_CSV = None
MIN_REVIEW_STARS = 0

# Approximate BRCA1 landmarks. Edit freely if you want a stricter domain map.
BRCA1_DOMAINS = pd.DataFrame([
    {"name": "RING", "start": 24, "end": 64},
    {"name": "DNA-binding region", "start": 452, "end": 1079},
    {"name": "Coiled-coil", "start": 1364, "end": 1437},
    {"name": "BRCT1", "start": 1646, "end": 1736},
    {"name": "BRCT2", "start": 1760, "end": 1855},
])

NOTEBOOK_PATCH_VERSION = "annotation-normalization-v2"
PATHOGENICITY_LABELS = {"benign", "pathogenic"}
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY*")

def ensure_plotting():
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
    return plt, sns

def read_csv(path, **kwargs):
    path = Path(path)
    return pd.read_csv(path, **kwargs) if path.is_file() else pd.DataFrame()

def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def coalesce_first_present(df, candidate_columns, default=np.nan):
    out = pd.Series(default, index=df.index, dtype=object)
    for column in candidate_columns:
        if column in df.columns:
            values = df[column]
            if isinstance(values, pd.DataFrame):
                values = values.iloc[:, 0]
            out = out.combine_first(values)
    return out

def normalize_annotation_columns(df):
    df = df.copy()
    annotation_candidates = [
        "annotation_active_label", "annotation_y", "clinvar_annotation",
        "annotation", "annotation_x",
    ]
    star_candidates = [
        "stars_active_label", "stars_y", "clinvar_review_stars",
        "stars", "stars_x",
    ]
    df["annotation"] = coalesce_first_present(df, annotation_candidates)
    df["annotation"] = df["annotation"].astype(str).str.lower().replace({"nan": np.nan, "none": np.nan, "": np.nan})
    df["stars"] = safe_numeric(coalesce_first_present(df, star_candidates, default=0)).fillna(0).astype(int)
    return df

def add_annotation_plot_column(df):
    df = normalize_annotation_columns(df)
    df["annotation_plot"] = df["annotation"].fillna("unlabeled")
    return df

def load_pickle(path):
    with Path(path).open("rb") as handle:
        return pickle.load(handle)

print(f"Notebook patch: {NOTEBOOK_PATCH_VERSION}")
print(f"Repo: {REPO_ROOT}")
print(f"Dataset: {DATASET}")
print(f"Figures: {FIGURE_DIR}")


## Find The Best Current-Run DeltaEmbSAE

This prefers current benchmark result CSVs over collected global tables, filters to the sequence-deduplicated run label, and selects the best single SAE by direction-normalized AUC.

In [ ]:
def load_all_benchmark_result_rows():
    result_dir = JOB_DIR / "benchmark_row_analysis" / "results"
    frames = []
    for path in sorted(result_dir.glob("*.csv")):
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if df.empty:
            continue
        df["source_result_path"] = str(path)
        frames.append(df)
    if frames:
        return pd.concat(frames, ignore_index=True, sort=False)
    return pd.DataFrame()

def load_best_sae_row(run_label=RUN_LABEL):
    metrics = load_all_benchmark_result_rows()
    if metrics.empty:
        metrics = read_csv(TABLE_DIR / f"{DATASET}_method_metrics.csv")
    if metrics.empty:
        metrics = read_csv(OUTPUT_ROOT / "tables" / "clinprotgym_method_metrics.csv")
    if metrics.empty:
        raise FileNotFoundError("No benchmark metrics/result rows found. Run DeltaEmbSAE benchmark jobs first.")
    if "dataset" in metrics.columns:
        metrics = metrics[metrics["dataset"].astype(str).eq(DATASET)].copy()
    fixed = metrics[metrics["benchmark"].astype(str).eq("Fixed DeltaEmbSAE")].copy()
    fixed = fixed[fixed["model_label"].astype(str).str.contains(run_label, regex=False, na=False)].copy()
    if fixed.empty:
        raise ValueError(f"No Fixed DeltaEmbSAE benchmark rows found for run label {run_label!r}.")
    fixed["spearman_rho"] = safe_numeric(fixed["spearman_rho"])
    fixed["auc"] = safe_numeric(fixed["auc"])
    fixed["auc_discrimination"] = fixed["auc"].map(lambda x: max(x, 1 - x) if pd.notna(x) else np.nan)
    fixed = fixed.sort_values(["auc_discrimination", "spearman_rho"], ascending=[False, False]).reset_index(drop=True)
    return fixed, fixed.iloc[0]

sae_candidates, best_sae_row = load_best_sae_row()
display(sae_candidates[[c for c in ["model_label", "model_short", "layer", "spearman_rho", "auc", "auc_discrimination", "feature_path", "fitness_path"] if c in sae_candidates.columns]].head(12))
print("Best SAE:")
display(best_sae_row[[c for c in ["model_label", "model_short", "layer", "spearman_rho", "auc", "auc_discrimination", "feature_path", "fitness_path"] if c in best_sae_row.index]])


## Load Mutation Metadata, Labels, Fitness, And Activations

In [ ]:
state = load_pickle(SEQUENCE_DIR / f"{DATASET}_processed_state.pkl")
metadata = state["sequence_metadata"].copy()
metadata["SequenceIndex"] = metadata["SequenceIndex"].astype(str)
metadata["position"] = safe_numeric(metadata["position"]).astype("Int64")
metadata["functional_score"] = safe_numeric(metadata["functional_score"])

scores_df = state["scores_dataframe"].copy()
scores_df["SequenceIndex"] = scores_df["SequenceIndex"].astype(str)

pipeline_labels = state["annotations_dataframe"].copy()
pipeline_labels["SequenceIndex"] = pipeline_labels["SequenceIndex"].astype(str)
pipeline_labels["annotation"] = pipeline_labels["annotation"].astype(str).str.lower().replace({"nan": np.nan})
if "stars" not in pipeline_labels.columns:
    pipeline_labels["stars"] = pipeline_labels.get("clinvar_review_stars", 0)
pipeline_labels["stars"] = safe_numeric(pipeline_labels["stars"]).fillna(0).astype(int)

def load_hgvs_labels():
    paths = [
        OUTPUT_ROOT / "tables" / "hgvs_clinvar_annotations" / f"{DATASET}_hgvs_clinvar_annotations.csv",
        TABLE_DIR / f"{DATASET}_hgvs_clinvar_annotations.csv",
    ]
    path = next((p for p in paths if p.is_file()), None)
    if path is None:
        raise FileNotFoundError("No HGVS ClinVar cache found. Run the downstream AUC refresh notebook/script first.")
    labels = pd.read_csv(path, dtype={"hgvs_nt": str}).rename(columns={"hgvs_nt": "SequenceIndex", "clinvar_review_stars": "stars"})
    labels["SequenceIndex"] = labels["SequenceIndex"].astype(str)
    labels["annotation"] = labels["annotation"].astype(str).str.lower().replace({"nan": np.nan})
    labels["stars"] = safe_numeric(labels.get("stars", 0)).fillna(0).astype(int)
    return labels[["SequenceIndex", "annotation", "stars"]].drop_duplicates("SequenceIndex", keep="first")

def load_custom_labels(path):
    labels = pd.read_csv(path)
    if "SequenceIndex" not in labels.columns:
        for candidate in ["hgvs_nt", "mutant", "protein_sequence_index"]:
            if candidate in labels.columns:
                labels = labels.rename(columns={candidate: "SequenceIndex"})
                break
    if "SequenceIndex" not in labels.columns or "annotation" not in labels.columns:
        raise ValueError("Custom labels need SequenceIndex and annotation columns.")
    if "stars" not in labels.columns:
        labels["stars"] = labels.get("clinvar_review_stars", 0)
    labels["SequenceIndex"] = labels["SequenceIndex"].astype(str)
    labels["annotation"] = labels["annotation"].astype(str).str.lower().replace({"nan": np.nan})
    labels["stars"] = safe_numeric(labels["stars"]).fillna(0).astype(int)
    return labels[["SequenceIndex", "annotation", "stars"]].drop_duplicates("SequenceIndex", keep="first")

if CLINICAL_LABEL_MODE == "pipeline":
    labels = pipeline_labels[["SequenceIndex", "annotation", "stars"]].drop_duplicates("SequenceIndex", keep="first")
elif CLINICAL_LABEL_MODE == "hgvs":
    labels = load_hgvs_labels()
elif CLINICAL_LABEL_MODE == "custom_csv":
    labels = load_custom_labels(CUSTOM_CLINICAL_LABELS_CSV)
else:
    raise ValueError(CLINICAL_LABEL_MODE)

labels = labels[labels["stars"].ge(MIN_REVIEW_STARS)].copy()

feature_path = Path(str(best_sae_row["feature_path"]))
seq_to_features = load_pickle(feature_path)
sequence_ids = list(seq_to_features)
activation_matrix = np.vstack([np.asarray(seq_to_features[seq_id], dtype=np.float32) for seq_id in sequence_ids])
activation_df_index = pd.Index(sequence_ids, name="SequenceIndex")

mutation_df = pd.DataFrame({"SequenceIndex": sequence_ids}).merge(metadata, on="SequenceIndex", how="left")
mutation_df = mutation_df.merge(labels, on="SequenceIndex", how="left", suffixes=("", "_active_label"))
mutation_df = normalize_annotation_columns(mutation_df)
mutation_df["functional_score"] = safe_numeric(mutation_df["functional_score"])
mutation_df["fitness"] = safe_numeric(read_csv(best_sae_row["fitness_path"]).set_index("SequenceIndex").reindex(sequence_ids)["fitness"].to_numpy()) if Path(str(best_sae_row["fitness_path"])).is_file() else np.nan

print(f"Activation matrix: {activation_matrix.shape[0]} mutations x {activation_matrix.shape[1]} SAE features")
print(f"Feature path: {feature_path}")
display(mutation_df[["SequenceIndex", "mutant", "position", "wt_aa", "mutant_aa", "functional_score", "annotation", "stars"]].head())
display(mutation_df["annotation"].value_counts(dropna=False).rename_axis("annotation").reset_index(name="n"))


## SAE Model And Reconstruction Diagnostics

This section is the SAE health check: model dimensions, reconstruction accuracy, train/test split, activation sparsity, feature utilization, and any dead/unused units. The exported activation matrix may contain only active SAE features; when that happens, `exported_feature_to_full_feature` maps notebook feature columns back to the original 12,800-unit SAE feature ids.

In [ ]:
def locate_sae_artifacts(feature_path):
    feature_path = Path(feature_path)
    run_dir = feature_path.parent
    model_dir = run_dir / "sae_models"
    viz_paths = sorted(model_dir.glob("*_viz_data.pkl"))
    model_paths = sorted(model_dir.glob("*_model.pt"))
    return {
        "run_dir": run_dir,
        "model_dir": model_dir,
        "viz_path": viz_paths[0] if viz_paths else None,
        "model_path": model_paths[0] if model_paths else None,
    }

sae_artifacts = locate_sae_artifacts(feature_path)
if sae_artifacts["viz_path"] is None:
    raise FileNotFoundError(f"Could not find SAE viz_data.pkl under {sae_artifacts['model_dir']}")
sae_viz = load_pickle(sae_artifacts["viz_path"])

X_original = np.asarray(sae_viz["X_original"], dtype=np.float32)
X_reconstructed = np.asarray(sae_viz["X_reconstructed"], dtype=np.float32)
Z_all_full = np.asarray(sae_viz["Z_all"], dtype=np.float32)
active_mask = np.asarray(sae_viz.get("active_mask", np.ones(Z_all_full.shape[1], dtype=bool))).astype(bool)

if activation_matrix.shape[1] == int(active_mask.sum()):
    exported_feature_to_full_feature = np.flatnonzero(active_mask).astype(int)
    feature_index_note = "exported activation columns are active SAE units only"
elif activation_matrix.shape[1] == Z_all_full.shape[1]:
    exported_feature_to_full_feature = np.arange(Z_all_full.shape[1], dtype=int)
    feature_index_note = "exported activation columns match full SAE feature ids"
else:
    exported_feature_to_full_feature = np.arange(activation_matrix.shape[1], dtype=int)
    feature_index_note = "could not infer full-feature mapping from active_mask; using exported column ids"

seq_ids_viz = [str(value) for value in sae_viz.get("seq_ids", [])]
train_idx = np.asarray(sae_viz.get("train_idx", []), dtype=int)
test_idx = np.asarray(sae_viz.get("test_idx", []), dtype=int)

residual = X_original - X_reconstructed
sample_mse = np.mean(residual ** 2, axis=1)
sample_mae = np.mean(np.abs(residual), axis=1)
sample_l2 = np.linalg.norm(residual, axis=1)
original_l2 = np.linalg.norm(X_original, axis=1)
relative_l2 = np.divide(sample_l2, original_l2, out=np.full_like(sample_l2, np.nan, dtype=float), where=original_l2 > 0)
cosine = np.sum(X_original * X_reconstructed, axis=1) / (
    np.linalg.norm(X_original, axis=1) * np.linalg.norm(X_reconstructed, axis=1) + 1e-12
)

def global_r2(x, xhat, row_idx=None):
    if row_idx is not None and len(row_idx):
        x = x[row_idx]
        xhat = xhat[row_idx]
    ss_res = float(np.sum((x - xhat) ** 2))
    ss_tot = float(np.sum((x - np.mean(x, axis=0, keepdims=True)) ** 2))
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

def mean_if_index(values, row_idx):
    return float(np.nanmean(values[row_idx])) if len(row_idx) else np.nan

activation_positive = activation_matrix > 0
variant_n_active = activation_positive.sum(axis=1)
feature_n_active = activation_positive.sum(axis=0)
feature_max_activation = activation_matrix.max(axis=0)
feature_mean_nonzero = np.divide(
    activation_matrix.sum(axis=0),
    feature_n_active,
    out=np.zeros(activation_matrix.shape[1], dtype=float),
    where=feature_n_active > 0,
)

full_variant_n_active = (Z_all_full > 0).sum(axis=1)
full_feature_n_active = (Z_all_full > 0).sum(axis=0)

sae_diagnostics = pd.DataFrame([
    {"category": "selected model", "metric": "model_label", "value": str(best_sae_row.get("model_label", ""))},
    {"category": "selected model", "metric": "benchmark_auc", "value": float(best_sae_row.get("auc", np.nan))},
    {"category": "selected model", "metric": "benchmark_spearman_rho", "value": float(best_sae_row.get("spearman_rho", np.nan))},
    {"category": "artifacts", "metric": "feature_path", "value": str(feature_path)},
    {"category": "artifacts", "metric": "viz_path", "value": str(sae_artifacts["viz_path"])},
    {"category": "artifacts", "metric": "model_path", "value": str(sae_artifacts["model_path"])},
    {"category": "dimensions", "metric": "n_variants_exported", "value": int(activation_matrix.shape[0])},
    {"category": "dimensions", "metric": "embedding_dim", "value": int(X_original.shape[1])},
    {"category": "dimensions", "metric": "n_full_sae_features", "value": int(Z_all_full.shape[1])},
    {"category": "dimensions", "metric": "n_exported_features", "value": int(activation_matrix.shape[1])},
    {"category": "dimensions", "metric": "n_active_mask_features", "value": int(active_mask.sum())},
    {"category": "dimensions", "metric": "n_masked_dead_features", "value": int((~active_mask).sum())},
    {"category": "dimensions", "metric": "feature_index_note", "value": feature_index_note},
    {"category": "split", "metric": "n_train", "value": int(len(train_idx))},
    {"category": "split", "metric": "n_test", "value": int(len(test_idx))},
    {"category": "split", "metric": "deduplicate_training_sequences", "value": bool(sae_viz.get("deduplicate_training_sequences", False))},
    {"category": "split", "metric": "n_training_input_sequences", "value": int(sae_viz.get("n_training_input_sequences", np.nan))},
    {"category": "split", "metric": "n_training_unique_protein_sequences", "value": int(sae_viz.get("n_training_unique_protein_sequences", np.nan))},
    {"category": "split", "metric": "n_training_duplicate_protein_sequences", "value": int(sae_viz.get("n_training_duplicate_protein_sequences", np.nan))},
    {"category": "reconstruction", "metric": "overall_mse", "value": float(np.mean(sample_mse))},
    {"category": "reconstruction", "metric": "overall_rmse", "value": float(np.sqrt(np.mean(sample_mse)))},
    {"category": "reconstruction", "metric": "overall_mae", "value": float(np.mean(sample_mae))},
    {"category": "reconstruction", "metric": "global_r2", "value": float(global_r2(X_original, X_reconstructed))},
    {"category": "reconstruction", "metric": "train_mse", "value": mean_if_index(sample_mse, train_idx)},
    {"category": "reconstruction", "metric": "test_mse", "value": mean_if_index(sample_mse, test_idx)},
    {"category": "reconstruction", "metric": "train_r2", "value": float(global_r2(X_original, X_reconstructed, train_idx)) if len(train_idx) else np.nan},
    {"category": "reconstruction", "metric": "test_r2", "value": float(global_r2(X_original, X_reconstructed, test_idx)) if len(test_idx) else np.nan},
    {"category": "reconstruction", "metric": "median_cosine_similarity", "value": float(np.nanmedian(cosine))},
    {"category": "reconstruction", "metric": "median_relative_l2_error", "value": float(np.nanmedian(relative_l2))},
    {"category": "sparsity", "metric": "exported_activation_density", "value": float(activation_positive.mean())},
    {"category": "sparsity", "metric": "median_active_exported_features_per_variant", "value": float(np.median(variant_n_active))},
    {"category": "sparsity", "metric": "mean_active_exported_features_per_variant", "value": float(np.mean(variant_n_active))},
    {"category": "sparsity", "metric": "max_active_exported_features_per_variant", "value": int(np.max(variant_n_active))},
    {"category": "sparsity", "metric": "n_exported_features_used_at_least_once", "value": int((feature_n_active > 0).sum())},
    {"category": "sparsity", "metric": "n_exported_features_never_used", "value": int((feature_n_active == 0).sum())},
    {"category": "sparsity", "metric": "full_activation_density", "value": float((Z_all_full > 0).mean())},
    {"category": "sparsity", "metric": "median_active_full_features_per_variant", "value": float(np.median(full_variant_n_active))},
    {"category": "sparsity", "metric": "n_full_features_used_at_least_once", "value": int((full_feature_n_active > 0).sum())},
    {"category": "training", "metric": "n_epochs", "value": int(len(sae_viz.get("train_losses", [])))},
    {"category": "training", "metric": "final_train_loss", "value": float(sae_viz.get("train_losses", [np.nan])[-1]) if sae_viz.get("train_losses") else np.nan},
    {"category": "training", "metric": "final_test_loss", "value": float(sae_viz.get("test_losses", [np.nan])[-1]) if sae_viz.get("test_losses") else np.nan},
])

sae_diagnostics_path = TABLE_OUT_DIR / f"{DATASET}_best_auc_sae_diagnostics.csv"
sae_diagnostics.to_csv(sae_diagnostics_path, index=False)
print(sae_diagnostics_path)
display(sae_diagnostics)


## SAE Diagnostics Plots

Training curves, reconstruction-error distributions, and sparsity/utilization plots. These are boring when the model is healthy, and very loud when something is off.

In [ ]:
plt, sns = ensure_plotting()

reconstruction_df = pd.DataFrame({
    "SequenceIndex": seq_ids_viz if len(seq_ids_viz) == len(sample_mse) else np.arange(len(sample_mse)).astype(str),
    "sample_mse": sample_mse,
    "sample_mae": sample_mae,
    "relative_l2_error": relative_l2,
    "cosine_similarity": cosine,
    "active_full_features": full_variant_n_active,
})
mutation_metadata_for_reconstruction = normalize_annotation_columns(mutation_df)
reconstruction_columns = [
    c for c in ["SequenceIndex", "mutant", "position", "functional_score", "annotation", "stars"]
    if c in mutation_metadata_for_reconstruction.columns
]
reconstruction_df = reconstruction_df.merge(
    mutation_metadata_for_reconstruction[reconstruction_columns],
    on="SequenceIndex",
    how="left",
)
reconstruction_df = normalize_annotation_columns(reconstruction_df)
reconstruction_df["split"] = "not_in_split"
if len(seq_ids_viz) == len(sample_mse):
    reconstruction_df.loc[train_idx, "split"] = "train"
    reconstruction_df.loc[test_idx, "split"] = "test"

feature_utilization_df = pd.DataFrame({
    "exported_feature": np.arange(activation_matrix.shape[1], dtype=int),
    "full_sae_feature": exported_feature_to_full_feature[:activation_matrix.shape[1]],
    "n_active_variants": feature_n_active,
    "fraction_active_variants": feature_n_active / activation_matrix.shape[0],
    "max_activation": feature_max_activation,
    "mean_nonzero_activation": feature_mean_nonzero,
})

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

loss_df = pd.DataFrame({
    "epoch": np.arange(1, len(sae_viz.get("train_losses", [])) + 1),
    "train_loss": sae_viz.get("train_losses", []),
    "test_loss": sae_viz.get("test_losses", []),
})
if not loss_df.empty:
    axes[0].plot(loss_df["epoch"], loss_df["train_loss"], label="train", linewidth=2)
    axes[0].plot(loss_df["epoch"], loss_df["test_loss"], label="test", linewidth=2)
    axes[0].set_yscale("log")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title("SAE training curve")
    axes[0].legend(frameon=False)
else:
    axes[0].text(0.5, 0.5, "No loss history", ha="center", va="center")

sns.histplot(data=reconstruction_df, x="sample_mse", hue="split", bins=40, element="step", stat="count", common_norm=False, ax=axes[1])
axes[1].set_xlabel("per-variant reconstruction MSE")
axes[1].set_title("Reconstruction error distribution")

sns.histplot(reconstruction_df["active_full_features"], bins=50, ax=axes[2], color="#4C78A8")
axes[2].set_xlabel("active full SAE features per variant")
axes[2].set_title("Per-variant sparsity")

sns.histplot(feature_utilization_df["n_active_variants"], bins=60, ax=axes[3], color="#54A24B")
axes[3].set_xlabel("variants activating feature")
axes[3].set_yscale("log")
axes[3].set_title("Feature utilization")

fig.tight_layout()
out = FIGURE_DIR / f"{DATASET}_best_auc_sae_training_reconstruction_sparsity.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(out)

reconstruction_path = TABLE_OUT_DIR / f"{DATASET}_best_auc_sae_reconstruction_by_variant.csv"
feature_utilization_path = TABLE_OUT_DIR / f"{DATASET}_best_auc_sae_feature_utilization.csv"
reconstruction_df.to_csv(reconstruction_path, index=False)
feature_utilization_df.to_csv(feature_utilization_path, index=False)
print(reconstruction_path)
print(feature_utilization_path)


## Reconstruction Outliers

Variants with high reconstruction error can be biologically interesting, but they can also flag regions where the SAE basis is doing a poor job. This table and plot help separate the two.

In [ ]:
plt, sns = ensure_plotting()

reconstruction_df = normalize_annotation_columns(reconstruction_df)
outlier_cols = [
    "SequenceIndex", "mutant", "position", "functional_score", "annotation", "stars", "split",
    "sample_mse", "relative_l2_error", "cosine_similarity", "active_full_features",
]
reconstruction_outliers = reconstruction_df.sort_values("sample_mse", ascending=False).head(40)
display(reconstruction_outliers[[c for c in outlier_cols if c in reconstruction_outliers.columns]])

plot_df = reconstruction_df[np.isfinite(reconstruction_df["position"].astype(float))].copy()
plot_df = add_annotation_plot_column(plot_df)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
sns.scatterplot(data=plot_df, x="position", y="sample_mse", hue="annotation_plot", alpha=0.55, s=28, edgecolor="none", ax=axes[0])
axes[0].set_xlabel("BRCA1 protein position")
axes[0].set_ylabel("reconstruction MSE")
axes[0].set_title("Reconstruction error by position")
axes[0].legend(title="Clinical label", frameon=False)

sns.scatterplot(data=plot_df, x="functional_score", y="sample_mse", hue="annotation_plot", alpha=0.55, s=28, edgecolor="none", ax=axes[1])
axes[1].set_xlabel("functional score")
axes[1].set_ylabel("reconstruction MSE")
axes[1].set_title("Reconstruction error vs functional score")
axes[1].legend(title="Clinical label", frameon=False)
fig.tight_layout()
out = FIGURE_DIR / f"{DATASET}_best_auc_sae_reconstruction_outliers.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(out)


## Feature Summary And Automatic Feature Selection

This ranks features by simple activation statistics plus optional biological signals. `coverage_score` favors features that are both active in multiple variants and have nontrivial activation magnitude.

In [ ]:
def auc_from_scores(scores, labels_bool):
    scores = np.asarray(scores, dtype=float)
    labels_bool = np.asarray(labels_bool, dtype=bool)
    finite = np.isfinite(scores)
    scores = scores[finite]
    labels_bool = labels_bool[finite]
    n_pos = int(labels_bool.sum())
    n_neg = int((~labels_bool).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = pd.Series(scores).rank(method="average").to_numpy()
    return float((ranks[labels_bool].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

def clinical_annotation_series(df):
    for column in ["annotation", "annotation_active_label", "annotation_y", "annotation_x", "clinvar_annotation"]:
        if column in df.columns:
            values = df[column]
            if isinstance(values, pd.DataFrame):
                values = values.iloc[:, 0]
            return values.astype(str).str.lower().replace({"nan": np.nan, "none": np.nan, "": np.nan})
    return pd.Series(np.nan, index=df.index, dtype=object)

def summarize_features(matrix, mutation_df, candidate_limit=TOP_FEATURES_TO_SUMMARIZE):
    mutation_df = mutation_df.copy()
    annotation_series = clinical_annotation_series(mutation_df)
    if "annotation" not in mutation_df.columns:
        mutation_df["annotation"] = annotation_series
    if "functional_score" not in mutation_df.columns:
        mutation_df["functional_score"] = np.nan
    nnz = np.count_nonzero(matrix > 0, axis=0)
    max_act = matrix.max(axis=0)
    mean_act = matrix.mean(axis=0)
    sum_act = matrix.sum(axis=0)
    mean_nonzero = np.divide(sum_act, nnz, out=np.zeros_like(sum_act, dtype=float), where=nnz > 0)
    coverage_score = np.log1p(nnz) * max_act
    summary = pd.DataFrame({
        "feature": np.arange(matrix.shape[1], dtype=int),
        "n_active": nnz,
        "fraction_active": nnz / matrix.shape[0],
        "max_activation": max_act,
        "mean_activation": mean_act,
        "mean_nonzero_activation": mean_nonzero,
        "coverage_score": coverage_score,
    })

    # Position concentration: among active variants, which residue position is most represented?
    positions = safe_numeric(mutation_df["position"]).to_numpy(dtype=float)
    top_positions = []
    top_position_counts = []
    n_position_values = []
    for feature_idx in summary.sort_values("coverage_score", ascending=False)["feature"].head(candidate_limit):
        active = matrix[:, feature_idx] > 0
        pos = positions[active]
        pos = pos[np.isfinite(pos)]
        if len(pos) == 0:
            top_positions.append((feature_idx, np.nan, 0, 0))
            continue
        counts = pd.Series(pos.astype(int)).value_counts()
        top_positions.append((feature_idx, int(counts.index[0]), int(counts.iloc[0]), int(counts.size)))
    pos_df = pd.DataFrame(top_positions, columns=["feature", "top_position", "top_position_active_count", "n_active_positions"])
    summary = summary.merge(pos_df, on="feature", how="left")

    # Lightweight clinical and functional association for top candidate features.
    annotation_series = clinical_annotation_series(mutation_df)
    binary = annotation_series.isin(PATHOGENICITY_LABELS).to_numpy()
    is_pathogenic = annotation_series.eq("pathogenic").to_numpy()
    functional_score = safe_numeric(mutation_df.get("functional_score", pd.Series(np.nan, index=mutation_df.index))).to_numpy(dtype=float)
    assoc_rows = []
    candidate_features = summary.sort_values("coverage_score", ascending=False)["feature"].head(candidate_limit).tolist()
    for feature_idx in candidate_features:
        values = matrix[:, feature_idx].astype(float)
        binary_values = values[binary]
        binary_path = is_pathogenic[binary]
        auc = auc_from_scores(binary_values, binary_path) if len(binary_values) else np.nan
        active = values > 0
        active_binary = active & binary
        inactive_binary = (~active) & binary
        a = int((active_binary & is_pathogenic).sum())
        b = int((active_binary & ~is_pathogenic).sum())
        c = int((inactive_binary & is_pathogenic).sum())
        d = int((inactive_binary & ~is_pathogenic).sum())
        oddsratio, pvalue = fisher_exact([[a, b], [c, d]]) if (a + b) and (c + d) else (np.nan, np.nan)
        finite = np.isfinite(values) & np.isfinite(functional_score)
        rho = spearmanr(values[finite], functional_score[finite]).statistic if finite.sum() >= 3 and np.nanstd(values[finite]) > 0 else np.nan
        assoc_rows.append({
            "feature": feature_idx,
            "auc_pathogenic_from_activation": auc,
            "auc_discrimination": max(auc, 1 - auc) if pd.notna(auc) else np.nan,
            "pathogenic_active": a,
            "benign_active": b,
            "pathogenic_inactive": c,
            "benign_inactive": d,
            "fisher_oddsratio": oddsratio,
            "fisher_pvalue": pvalue,
            "spearman_activation_functional_score": rho,
            "spearman_abs": abs(rho) if pd.notna(rho) else np.nan,
        })
    assoc_df = pd.DataFrame(assoc_rows)
    summary = summary.merge(assoc_df, on="feature", how="left")
    return summary

feature_summary = summarize_features(activation_matrix, mutation_df)
if "exported_feature_to_full_feature" in globals():
    feature_summary["full_sae_feature"] = exported_feature_to_full_feature[feature_summary["feature"].astype(int).to_numpy()]
feature_summary_path = TABLE_OUT_DIR / f"{DATASET}_best_auc_sae_feature_summary.csv"
feature_summary.to_csv(feature_summary_path, index=False)

sort_col = FEATURE_SELECTION_METRIC if FEATURE_SELECTION_METRIC in feature_summary.columns else "coverage_score"
feature_summary_sorted = feature_summary.sort_values([sort_col, "coverage_score"], ascending=[False, False]).reset_index(drop=True)
selected_features = feature_summary_sorted["feature"].head(TOP_FEATURES_TO_PLOT).astype(int).tolist()
if SELECTED_FEATURE is None:
    SELECTED_FEATURE = int(selected_features[0])

print(f"Feature summary: {feature_summary_path}")
print(f"Selected feature for detailed plots: {SELECTED_FEATURE}")
display(feature_summary_sorted.head(20))


## Feature Activation Along The Protein Sequence

This is the core view: protein residue position on x, activation on y. Each point is a mutation. Color is clinical annotation by default.

In [ ]:
plt, sns = ensure_plotting()

def add_brca1_domain_bands(ax, y_top=None):
    ymin, ymax = ax.get_ylim()
    label_y = ymax if y_top is None else y_top
    for _, domain in BRCA1_DOMAINS.iterrows():
        ax.axvspan(domain["start"], domain["end"], color="0.85", alpha=0.35, zorder=0)
        ax.text((domain["start"] + domain["end"]) / 2, label_y, domain["name"], ha="center", va="top", fontsize=8, color="0.35", rotation=0)
    ax.set_ylim(ymin, ymax)

def activation_long_for_features(features):
    rows = []
    base_mutation_df = normalize_annotation_columns(mutation_df)
    for feature_idx in features:
        df = base_mutation_df.copy()
        df["feature"] = int(feature_idx)
        df["activation"] = activation_matrix[:, int(feature_idx)]
        rows.append(df)
    return pd.concat(rows, ignore_index=True, sort=False)

selected_df = activation_long_for_features([SELECTED_FEATURE])
plot_df = selected_df[np.isfinite(selected_df["position"].astype(float))].copy()
plot_df = add_annotation_plot_column(plot_df)

fig, ax = plt.subplots(figsize=(13, 5.4))
sns.scatterplot(
    data=plot_df,
    x="position",
    y="activation",
    hue="annotation_plot",
    style="annotation_plot",
    alpha=0.70,
    s=42,
    edgecolor="none",
    ax=ax,
)
add_brca1_domain_bands(ax)
ax.set_xlabel("BRCA1 protein position")
ax.set_ylabel(f"SAE feature {SELECTED_FEATURE} activation")
ax.set_title(f"{best_sae_row['model_label']} | feature {SELECTED_FEATURE}")
ax.legend(title="Clinical label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

# Label the strongest activating mutations.
top = plot_df.sort_values("activation", ascending=False).head(12)
for _, row in top.iterrows():
    if row["activation"] <= 0:
        continue
    ax.annotate(str(row["mutant"]), (row["position"], row["activation"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
fig.tight_layout()
out = FIGURE_DIR / f"{DATASET}_feature_{SELECTED_FEATURE}_activation_by_position.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(out)


## Small Multiples: Top Features Along BRCA1

Same position-vs-activation view for the top selected features.

In [ ]:
plt, sns = ensure_plotting()
features_to_plot = selected_features[:TOP_FEATURES_TO_PLOT]
plot_df = activation_long_for_features(features_to_plot)
plot_df = plot_df[np.isfinite(plot_df["position"].astype(float))].copy()
plot_df["feature_label"] = "feature " + plot_df["feature"].astype(str)
plot_df = add_annotation_plot_column(plot_df)

ncols = 3
nrows = int(np.ceil(len(features_to_plot) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 3.3 * nrows), squeeze=False, sharex=True)
palette = {"pathogenic": "#D7301F", "benign": "#2C7FB8", "unlabeled": "#9A9A9A"}
for ax, feature_idx in zip(axes.ravel(), features_to_plot):
    sub = plot_df[plot_df["feature"].eq(feature_idx)]
    for label, group in sub.groupby("annotation_plot", dropna=False):
        ax.scatter(group["position"], group["activation"], s=13, alpha=0.42 if label == "unlabeled" else 0.78, color=palette.get(label, "0.5"), label=label, edgecolor="none")
    ax.set_title(f"feature {feature_idx}")
    ax.set_xlabel("position")
    ax.set_ylabel("activation")
    for _, domain in BRCA1_DOMAINS.iterrows():
        ax.axvspan(domain["start"], domain["end"], color="0.90", alpha=0.25, zorder=0)
for ax in axes.ravel()[len(features_to_plot):]:
    ax.axis("off")
handles, labels_ = axes.ravel()[0].get_legend_handles_labels()
fig.legend(handles, labels_, title="Clinical label", bbox_to_anchor=(1.01, 0.98), loc="upper left", frameon=False)
fig.suptitle(f"Top SAE features by {FEATURE_SELECTION_METRIC}: activation along BRCA1", y=1.02)
fig.tight_layout()
out = FIGURE_DIR / f"{DATASET}_top_sae_features_activation_by_position.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(out)


## Feature-By-Position Heatmap

For each selected feature and protein position, this aggregates the maximum activation across all substitutions at that position.

In [ ]:
plt, sns = ensure_plotting()
features_for_heatmap = selected_features[:min(30, len(selected_features))]
heat_rows = []
positions = safe_numeric(mutation_df["position"]).to_numpy(dtype=float)
for feature_idx in features_for_heatmap:
    values = activation_matrix[:, int(feature_idx)]
    tmp = pd.DataFrame({"position": positions, "activation": values}).dropna()
    tmp["position"] = tmp["position"].astype(int)
    by_pos = tmp.groupby("position")["activation"].max()
    for pos, activation in by_pos.items():
        heat_rows.append({"feature": int(feature_idx), "position": int(pos), "max_activation": float(activation)})
heat_df = pd.DataFrame(heat_rows)
if heat_df.empty:
    print("No heatmap rows available.")
else:
    pivot = heat_df.pivot_table(index="feature", columns="position", values="max_activation", aggfunc="max", fill_value=0.0)
    # Bin positions to make a readable heatmap across 1863 aa.
    bin_size = 20
    binned = []
    for feature_idx, row in pivot.iterrows():
        values = row.reset_index()
        values["bin_start"] = ((values["position"] - 1) // bin_size) * bin_size + 1
        for bin_start, group in values.groupby("bin_start"):
            binned.append({"feature": feature_idx, "bin_start": bin_start, "max_activation": group[feature_idx].max()})
    binned_df = pd.DataFrame(binned)
    binned_pivot = binned_df.pivot_table(index="feature", columns="bin_start", values="max_activation", fill_value=0.0)
    fig_width = max(12, binned_pivot.shape[1] * 0.14)
    fig_height = max(4, binned_pivot.shape[0] * 0.28)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    sns.heatmap(binned_pivot, cmap="mako", ax=ax, cbar_kws={"label": "max activation in 20-aa bin"})
    ax.set_xlabel("BRCA1 position bin start")
    ax.set_ylabel("SAE feature")
    ax.set_title("Selected SAE features: maximum activation by BRCA1 position")
    fig.tight_layout()
    out = FIGURE_DIR / f"{DATASET}_selected_features_position_heatmap.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(out)


## Top Mutations For Each Feature

A table-first view: the strongest activating substitutions for each selected feature, with functional scores and labels attached.

In [ ]:
top_rows = []
for feature_idx in selected_features:
    values = activation_matrix[:, int(feature_idx)]
    order = np.argsort(values)[::-1][:TOP_VARIANTS_PER_FEATURE]
    for rank, row_idx in enumerate(order, start=1):
        if values[row_idx] <= 0:
            continue
        row = mutation_df.iloc[row_idx].to_dict()
        top_rows.append({
            "feature": int(feature_idx),
            "activation_rank": rank,
            "activation": float(values[row_idx]),
            "SequenceIndex": row.get("SequenceIndex"),
            "mutant": row.get("mutant"),
            "position": row.get("position"),
            "wt_aa": row.get("wt_aa"),
            "mutant_aa": row.get("mutant_aa"),
            "functional_score": row.get("functional_score"),
            "annotation": row.get("annotation"),
            "stars": row.get("stars"),
        })
top_feature_mutations = pd.DataFrame(top_rows)
top_feature_mutations_path = TABLE_OUT_DIR / f"{DATASET}_top_mutations_by_sae_feature.csv"
top_feature_mutations.to_csv(top_feature_mutations_path, index=False)
print(top_feature_mutations_path)
display(top_feature_mutations.head(80))


## Mutant Amino-Acid Enrichment For One Feature

For the selected feature, this shows which substituted amino acids activate at which positions.

In [ ]:
plt, sns = ensure_plotting()
feature_idx = SELECTED_FEATURE
values = activation_matrix[:, int(feature_idx)]
aa_df = mutation_df.copy()
aa_df["activation"] = values
aa_df = aa_df[aa_df["activation"].gt(0) & aa_df["position"].notna() & aa_df["mutant_aa"].notna()].copy()
aa_df["position"] = safe_numeric(aa_df["position"]).astype(int)
# Keep positions with strongest activation to make the matrix readable.
top_positions = aa_df.groupby("position")["activation"].max().sort_values(ascending=False).head(35).index.tolist()
aa_df = aa_df[aa_df["position"].isin(top_positions)].copy()
if aa_df.empty:
    print(f"Feature {feature_idx} has no positive activations to plot.")
else:
    aa_pivot = aa_df.pivot_table(index="mutant_aa", columns="position", values="activation", aggfunc="max", fill_value=0.0)
    aa_pivot = aa_pivot.reindex([aa for aa in AA_ORDER if aa in aa_pivot.index])
    aa_pivot = aa_pivot[sorted(aa_pivot.columns)]
    fig, ax = plt.subplots(figsize=(max(9, 0.36 * aa_pivot.shape[1]), 6.2))
    sns.heatmap(aa_pivot, cmap="viridis", ax=ax, cbar_kws={"label": "max activation"})
    ax.set_xlabel("BRCA1 position")
    ax.set_ylabel("mutant amino acid")
    ax.set_title(f"Feature {feature_idx}: mutant amino-acid activation map")
    fig.tight_layout()
    out = FIGURE_DIR / f"{DATASET}_feature_{feature_idx}_mutant_aa_heatmap.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(out)


## Feature Activation Versus Functional Score

This checks whether a feature is behaving like a broad fitness/pathogenicity axis or something more local/specific.

In [ ]:
plt, sns = ensure_plotting()
feature_idx = SELECTED_FEATURE
plot_df = normalize_annotation_columns(mutation_df)
plot_df["activation"] = activation_matrix[:, int(feature_idx)]
plot_df = add_annotation_plot_column(plot_df)
finite = np.isfinite(plot_df["activation"]) & np.isfinite(plot_df["functional_score"])
rho = spearmanr(plot_df.loc[finite, "activation"], plot_df.loc[finite, "functional_score"]).statistic if finite.sum() >= 3 and plot_df.loc[finite, "activation"].std() > 0 else np.nan
fig, ax = plt.subplots(figsize=(7.4, 5.6))
sns.scatterplot(data=plot_df, x="activation", y="functional_score", hue="annotation_plot", alpha=0.58, s=35, edgecolor="none", ax=ax)
ax.axvline(0, color="0.8", linewidth=1)
ax.axhline(0, color="0.8", linewidth=1)
ax.set_xlabel(f"feature {feature_idx} activation")
ax.set_ylabel("BRCA1 functional score")
ax.set_title(f"Feature {feature_idx}: activation vs functional score (Spearman={rho:.3f})")
ax.legend(title="Clinical label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.tight_layout()
out = FIGURE_DIR / f"{DATASET}_feature_{feature_idx}_activation_vs_functional_score.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(out)


## Feature Coactivation Map

Looks for groups of SAE features that tend to fire on the same variants.

In [ ]:
plt, sns = ensure_plotting()
features_for_coactivation = selected_features[:min(30, len(selected_features))]
sub = activation_matrix[:, features_for_coactivation]
# Spearman correlation on activations; fill degenerate columns with zero correlations.
corr = pd.DataFrame(sub, columns=features_for_coactivation).corr(method="spearman").fillna(0.0)
fig, ax = plt.subplots(figsize=(8.4, 7.2))
sns.heatmap(corr, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "Spearman correlation"})
ax.set_xlabel("SAE feature")
ax.set_ylabel("SAE feature")
ax.set_title("Coactivation between selected SAE features")
fig.tight_layout()
out = FIGURE_DIR / f"{DATASET}_selected_sae_feature_coactivation.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print(out)


## Feature Enrichment Table

Sort this table by `auc_discrimination`, `fisher_pvalue`, `spearman_abs`, or `top_position_active_count` to find features worth inspecting manually.

In [ ]:
display_cols = [
    "feature", "n_active", "fraction_active", "max_activation", "coverage_score",
    "top_position", "top_position_active_count", "n_active_positions",
    "auc_pathogenic_from_activation", "auc_discrimination", "fisher_oddsratio", "fisher_pvalue",
    "spearman_activation_functional_score", "spearman_abs",
]
display(feature_summary.sort_values(["auc_discrimination", "coverage_score"], ascending=[False, False])[[c for c in display_cols if c in feature_summary.columns]].head(80))


## Exported Outputs

The notebook writes figures under `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/figures/brca1_sae_interpretability/` and tables under `data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018/tables/sae_interpretability/`.